In [ ]:

# Cell 1: Import download dependencies
import os
import zipfile
import requests
from io import BytesIO

# Cell 2: Create base directory
BASE_DIR = '/content/drive/MyDrive/Project'
os.makedirs(BASE_DIR, exist_ok=True)

# # Cell 3: Download Flickr8k images
# images_url = 'https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip'
# response = requests.get(images_url)
# zip_file = zipfile.ZipFile(BytesIO(response.content))
# zip_file.extractall(BASE_DIR)
IMAGE_DIR = os.path.join(BASE_DIR, 'Flicker8k_Dataset')  # Note: The extracted folder is 'Flicker8k_Dataset'

# # Cell 4: Download Flickr8k text
# text_url = 'https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip'
# response = requests.get(text_url)
# zip_file = zipfile.ZipFile(BytesIO(response.content))
# zip_file.extractall(BASE_DIR)

# Cell 5: Set paths for text files
CAPTION_FILE = os.path.join(BASE_DIR, 'Flickr8k.token.txt')
TRAIN_IMAGE_FILE = os.path.join(BASE_DIR, 'Flickr_8k.trainImages.txt')
TEST_IMAGE_FILE = os.path.join(BASE_DIR, 'Flickr_8k.testImages.txt')

# # Cell 6: Download GloVe
# glove_url = 'http://nlp.stanford.edu/data/glove.6B.zip'
# response = requests.get(glove_url)
# zip_file = zipfile.ZipFile(BytesIO(response.content))
# zip_file.extract('glove.6B.100d.txt', path=BASE_DIR)
GLOVE_FILE = os.path.join(BASE_DIR, 'glove.6B.100d.txt')

# Cell 7: Create features directory
FEATURE_DIR = os.path.join(BASE_DIR, 'features')
os.makedirs(FEATURE_DIR, exist_ok=True)

# Cell 8: Set other paths
TOKENIZER_PATH = os.path.join(BASE_DIR, 'tokenizer.pkl')
MODEL_PATH = os.path.join(BASE_DIR, 'model.h5')

# Cell 9: Import main libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Dense, Dropout, Embedding, RepeatVector
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from transformers import ViTImageProcessor, ViTModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction, corpus_bleu
import pickle
from tqdm import tqdm
import torch



In [ ]:
# Cell 10: Load ViT processor and model
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
vit_model = ViTModel.from_pretrained('google/vit-base-patch16-224')

# Cell 11: Define extract_feature function
def extract_feature(image_path):
    image = load_img(image_path, target_size=(224, 224))
    image = img_to_array(image)
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = vit_model(**inputs)
    feature = outputs.last_hidden_state[:, 0, :]  # CLS token
    return feature.squeeze().numpy()  # Shape: (768,)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Cell 12: Load train and test images lists
with open(TRAIN_IMAGE_FILE, 'r') as f:
    train_images = [line.strip() for line in f.readlines()]
with open(TEST_IMAGE_FILE, 'r') as f:
    test_images = [line.strip() for line in f.readlines()]
all_images = train_images + test_images

# Cell 13: Extract and save features
for img_name in tqdm(all_images, desc="Extracting features with ViT"):
    image_path = os.path.join(IMAGE_DIR, img_name)
    output_path = os.path.join(FEATURE_DIR, img_name + '.npy')
    if not os.path.exists(output_path) and os.path.exists(image_path):
        feature = extract_feature(image_path)
        np.save(output_path, feature)
    elif not os.path.exists(image_path):
        print(f"Image not found: {image_path}")

Extracting features with ViT: 100%|██████████| 7000/7000 [00:27<00:00, 250.59it/s]


In [ ]:
# Cell 14: Define load_captions function
def load_captions(caption_file):
    with open(caption_file, 'r') as f:
        lines = f.readlines()
    captions_dict = {}
    for line in lines:
        tokens = line.strip().split('\t')
        if len(tokens) < 2:
            continue
        image_id, caption = tokens[0].split('#')[0], tokens[1]
        caption = caption.lower()
        caption = ''.join(c if c.isalnum() or c.isspace() else ' ' for c in caption)
        if image_id not in captions_dict:
            captions_dict[image_id] = []
        captions_dict[image_id].append(caption)
    return captions_dict

# Cell 15: Define save_tokenizer function
def save_tokenizer(tokenizer, path=TOKENIZER_PATH):
    with open(path, 'wb') as handle:
        pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Cell 16: Define load_tokenizer function
def load_tokenizer(path=TOKENIZER_PATH):
    with open(path, 'rb') as handle:
        tokenizer = pickle.load(handle)
    return tokenizer

# Cell 17: Define prepare_data function
def prepare_data():
    captions_dict = load_captions(CAPTION_FILE)

    train_captions = []
    for img in train_images:
        if img in captions_dict:
            for caption in captions_dict[img]:
                train_captions.append(f"startseq {caption} endseq")

    tokenizer = Tokenizer(num_words=10000, oov_token="<unk>")
    tokenizer.fit_on_texts(train_captions)
    save_tokenizer(tokenizer)
    train_sequences = tokenizer.texts_to_sequences(train_captions)

    max_length = max(len(seq) for seq in train_sequences)
    max_length = min(max_length, 39)  # Fixed as per your code

    train_sequences = pad_sequences(train_sequences, maxlen=max_length, padding='post')

    test_captions = []
    for img in test_images:
        if img in captions_dict:
            for caption in captions_dict[img]:
                test_captions.append(f"startseq {caption} endseq")
    test_sequences = tokenizer.texts_to_sequences(test_captions)
    test_sequences = pad_sequences(test_sequences, maxlen=max_length, padding='post')

    train_features = []
    for img in tqdm(train_images, desc="Loading train features"):
        feature_path = os.path.join(FEATURE_DIR, img + '.npy')
        if os.path.exists(feature_path):
            feature = np.load(feature_path)
            train_features.append(feature)
    train_features = np.array(train_features)

    test_features = []
    for img in tqdm(test_images, desc="Loading test features"):
        feature_path = os.path.join(FEATURE_DIR, img + '.npy')
        if os.path.exists(feature_path):
            feature = np.load(feature_path)
            test_features.append(feature)
    test_features = np.array(test_features)

    embedding_dim = 100  # GloVe dim
    embedding_index = {}
    with open(GLOVE_FILE, encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embedding_index[word] = coefs

    vocab_size = len(tokenizer.word_index) + 1
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    for word, i in tokenizer.word_index.items():
        if i < vocab_size:
            embedding_vector = embedding_index.get(word)
            if embedding_vector is not None:
                embedding_matrix[i] = embedding_vector

    return train_features, train_sequences, test_features, test_sequences, tokenizer, max_length, vocab_size, embedding_matrix, embedding_dim

# Cell 18: Call prepare_data
train_features, train_sequences, test_features, test_sequences, tokenizer, max_length, vocab_size, embedding_matrix, embedding_dim = prepare_data()

Loading test features: 100%|██████████| 1000/1000 [06:06<00:00,  2.73it/s]


In [ ]:
# Cell 19: Import ops for model
from keras import ops
# Cell 20: Define TransformerDecoder class
class TransformerDecoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, rate=0.1, trainable=True, **kwargs):
        super(TransformerDecoder, self).__init__(trainable=trainable, **kwargs)
        self.num_layers = num_layers
        self.d_model = d_model
        self.num_heads = num_heads
        self.dff = dff
        self.rate = rate
        self.self_mha_layers = [tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model) for _ in range(num_layers)]
        self.cross_mha_layers = [tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model) for _ in range(num_layers)]
        self.ffn_layers = [tf.keras.Sequential([Dense(dff, activation='relu'), Dense(d_model)]) for _ in range(num_layers)]
        self.layernorm1_layers = [tf.keras.layers.LayerNormalization(epsilon=1e-6) for _ in range(num_layers)]
        self.layernorm2_layers = [tf.keras.layers.LayerNormalization(epsilon=1e-6) for _ in range(num_layers)]
        self.layernorm3_layers = [tf.keras.layers.LayerNormalization(epsilon=1e-6) for _ in range(num_layers)]
        self.dropout1_layers = [Dropout(rate) for _ in range(num_layers)]
        self.dropout2_layers = [Dropout(rate) for _ in range(num_layers)]
        self.dropout3_layers = [Dropout(rate) for _ in range(num_layers)]

    def call(self, x, enc_output, training, **kwargs):
        seq_len = ops.shape(x)[1]
        batch_size = ops.shape(x)[0]

        # Causal mask for self-attention
        causal_mask = ops.cast(ops.expand_dims(ops.arange(seq_len), 1) >= ops.arange(seq_len), 'bool')
        causal_mask = ops.tile(ops.expand_dims(causal_mask, 0), [batch_size, 1, 1])

        # Use the generated causal mask
        combined_mask = causal_mask

        for i in range(self.num_layers):
            # Self-attention
            self_attn_output = self.self_mha_layers[i](query=x, key=x, value=x, attention_mask=combined_mask)
            dropout1 = self.dropout1_layers[i](self_attn_output, training=training)
            out1 = self.layernorm1_layers[i](x + dropout1)

            # Cross-attention
            cross_attn_output = self.cross_mha_layers[i](query=out1, key=enc_output, value=enc_output)
            dropout2 = self.dropout2_layers[i](cross_attn_output, training=training)
            out2 = self.layernorm2_layers[i](out1 + dropout2)

            # FFN
            ffn_output = self.ffn_layers[i](out2)
            dropout3 = self.dropout3_layers[i](ffn_output, training=training)
            x = self.layernorm3_layers[i](out2 + dropout3)

        return x

    def get_config(self):
        config = super(TransformerDecoder, self).get_config()
        config.update({
            'num_layers': self.num_layers,
            'd_model': self.d_model,
            'num_heads': self.num_heads,
            'dff': self.dff,
            'rate': self.rate,
            'trainable': self.trainable,
        })
        return config

# Cell 21: Define create_model function
def create_model(vocab_size, max_length):
    d_model = 256  # Increased decoder dim
    inputs1 = tf.keras.Input(shape=(768,))  # ViT base output dim
    fe1 = Dense(d_model, activation='relu')(inputs1)  # Project to d_model
    fe2 = ops.expand_dims(fe1, axis=1)  # (batch, 1, d_model) for broadcasting in attention

    inputs2 = tf.keras.Input(shape=(max_length-1,))
    se1 = Embedding(vocab_size, d_model, mask_zero=True)(inputs2)
    se2 = Dropout(0.5)(se1)

    decoder = TransformerDecoder(num_layers=6, d_model=d_model, num_heads=8, dff=1024, rate=0.2)
    dec_output = decoder(se2, fe2, training=True)

    outputs = Dense(vocab_size, activation='softmax')(dec_output)

    return tf.keras.Model(inputs=[inputs1, inputs2], outputs=outputs)

# Cell 22: Create model and summary
model = create_model(vocab_size, max_length)
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'transformer_decoder' (of type TransformerDecoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 38)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer         │ (None, 768)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 38, 256)   │  1,887,744 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    196,864 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 38, 256)   │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (None, 1, 256)    │          0 │ dense[0][0]       │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_decoder │ (None, 38, 256)   │ 28,405,248 │ dropout[0][0],    │
│ (TransformerDecode… │                   │            │ expand_dims[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 38, 7374)  │  1,895,118 │ transformer_deco… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,384,974 (123.54 MB)

 Trainable params: 32,384,974 (123.54 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Cell 28: Define evaluate_model function
def evaluate_model(model, test_features, test_sequences, tokenizer, max_length):
    bleu_scores = []
    smoothing_fn = SmoothingFunction().method4

    for i in range(len(test_features)):
        feature = test_features[i]
        true_captions = test_sequences[i*5:(i+1)*5]  # 5 captions per image

        true_texts = []
        references = []
        for true_caption in true_captions:
            true_caption_text = ' '.join([tokenizer.index_word.get(word, '') for word in true_caption if word != 0])
            true_texts.append(true_caption_text)
            references.append(true_caption_text.split())

        pred_caption = beam_search_predict(model, feature, tokenizer, max_length)
        pred_split = pred_caption.split()

        # Tính BLEU-1 cho prediction so với list references
        bleu_score = sentence_bleu(references, pred_split, weights=(1, 0, 0, 0), smoothing_function=smoothing_fn)
        bleu_scores.append(bleu_score)

        # In true captions, prediction, và BLEU score
        print(f"Image {i+1}:")
        print("True Captions:")
        for txt in true_texts:
            print(f" - {txt}")
        print(f"Pred: {pred_caption}")
        print(f"BLEU-1: {bleu_score:.4f}\n")

    # Tính average BLEU-1
    avg_bleu = np.mean(bleu_scores)
    print(f"Average BLEU-1: {avg_bleu:.4f}")

In [ ]:
# Cell 23: Import ModelCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint

# Cell 24: Prepare training data
X1_train = np.repeat(train_features, 5, axis=0)
X2_train = train_sequences[:, :-1]
y_train = train_sequences[:, 1:]

X1_test = np.repeat(test_features, 5, axis=0)
X2_test = test_sequences[:, :-1]
y_test = test_sequences[:, 1:]

# Cell 25: Set callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
checkpoint = ModelCheckpoint(MODEL_PATH, monitor='val_loss', save_best_only=True, mode='min', verbose=1)

# Cell 26: Compile and train model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00003), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit([X1_train, X2_train], y_train, validation_data=([X1_test, X2_test], y_test), epochs=50, batch_size=64, callbacks=[early_stopping, lr_scheduler, checkpoint])


In [ ]:
# Cell 27: Define beam_search_predict function
def beam_search_predict(model, feature, tokenizer, max_length, beam_width=10):
    start_token = tokenizer.word_index['startseq']
    end_token = tokenizer.word_index.get('endseq', 0)  # Handle if not present

    sequences = [[ [start_token] , 0.0 ]]

    feature_expanded = np.expand_dims(feature, axis=0)

    for step in range(max_length):
        all_candidates = []

        for seq, score in sequences:
            if seq[-1] == end_token:
                all_candidates.append([seq, score])
                continue

            seq_padded = pad_sequences([seq], maxlen=max_length-1, padding='post')
            pred = model.predict([feature_expanded, seq_padded], verbose=0)
            pred = pred[0, min(step, pred.shape[1]-1), :]

            top_words = np.argsort(pred)[-beam_width:]
            for word in top_words:
                new_seq = seq + [word]
                new_score = score + np.log(pred[word] + 1e-10)
                all_candidates.append([new_seq, new_score])

        # Normalize scores by length to favor longer sequences
        all_candidates = sorted(all_candidates, key=lambda x: x[1] / (len(x[0]) ** 0.6), reverse=True)
        sequences = all_candidates[:beam_width]

        if all(seq[-1] == end_token for seq, _ in sequences):
            break

    # Select the best sequence
    sequences = sorted(sequences, key=lambda x: x[1] / (len(x[0]) ** 0.6), reverse=True)
    best_seq, _ = sequences[0]
    caption = ' '.join([tokenizer.index_word.get(word, '') for word in best_seq if word in tokenizer.index_word and word not in [start_token, end_token]])
    return caption



# Cell 29: Evaluate model
evaluate_model(model, test_features, test_sequences, tokenizer, max_length)


In [ ]:
!pip install gtts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1


In [ ]:
# Cell 11: Add Gradio for Demo88

import gradio as gr
from PIL import Image
from gtts import gTTS

# Load ViT processor and model
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
vit_model = ViTModel.from_pretrained('google/vit-base-patch16-224')

# Function to generate caption (fixed greedy decoding)
def generate_caption(feature, tokenizer, model, max_length):
    in_text = 'startseq'
    # Đảm bảo mô hình chạy ở chế độ inference
    model_inference = tf.keras.Model(inputs=model.inputs, outputs=model.outputs)

    for _ in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        current_len = len(sequence)

        # Nếu chuỗi đã đạt max_length - 1, dừng lại
        if current_len >= max_length:
            break

        sequence = pad_sequences([sequence], maxlen=max_length - 1, padding='post')

        # Dự đoán
        # Dùng model_inference.predict để tránh lỗi liên quan đến training=True trong custom layer
        yhat = model_inference.predict([feature, sequence], verbose=0)

        # Lấy dự đoán cho timestep hiện tại (index: current_len - 1)
        yhat_index = np.argmax(yhat[0, current_len - 1, :])
        word = tokenizer.index_word.get(yhat_index, None)

        if word is None or word == '<unk>':
            break

        in_text += ' ' + word
        if word == 'endseq':
            break

    return in_text.replace('startseq ', '').replace(' endseq', '')

# Gradio function with TTS
def caption_image(uploaded_image):
    if uploaded_image is None:
        return "Vui lòng upload một hình ảnh.", None

    # Extract feature
    image = Image.open(uploaded_image).convert('RGB').resize((224, 224))
    image = img_to_array(image)
    inputs = processor(images=image, return_tensors="pt")

    # Chạy ViT model (PyTorch)
    with torch.no_grad():
        outputs = vit_model(**inputs)

    feature = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
    feature = np.expand_dims(feature, axis=0) # (1, 768)

    # Generate caption
    caption = generate_caption(feature, tokenizer, model, max_length)

    # Generate audio from caption using gTTS
    audio_path = f"/tmp/caption_{uuid.uuid4().hex}.mp3" # Tên file độc nhất
    try:
        tts = gTTS(caption, lang='vi' if 'Tiếng Việt' in caption else 'en') # Thử đoán ngôn ngữ
        tts.save(audio_path)
    except Exception as e:
        print(f"Lỗi TTS: {e}")
        audio_path = None # Trả về None nếu TTS lỗi

    return caption, audio_path

# Import uuid để tạo tên file độc nhất (tránh lỗi cache)
import uuid

# Thiết lập theme cho Gradio (giữ nguyên)
theme = gr.themes.Monochrome()

with gr.Blocks(theme=theme) as demo:
    gr.Markdown("# 🖼️ Image Captioning ")
    # Bố cục 2 cột (giống Streamlit columns)
    with gr.Row():
        # Cột 1: Upload Ảnh
        with gr.Column(scale=1):
            image_input = gr.Image(type="filepath", label="1. Upload Hình ảnh (JPG/PNG)")

            # ĐÃ XÓA: submit_btn = gr.Button("✨ Tạo Caption", variant="primary")
            # Không cần nút bấm, vì nó sẽ chạy tự động.

        # Cột 2: Hiển thị Kết quả
        with gr.Column(scale=2):
            caption_output = gr.Textbox(label="2. Mô tả được tạo (Generated Caption)", interactive=False)
            audio_output = gr.Audio(label="3. Audio Mô tả (Caption Audio)", autoplay=True, interactive=False)

    # Gán sự kiện TỰ ĐỘNG CHẠY (Trigger)
    # Hàm caption_image sẽ được gọi ngay sau khi giá trị của image_input thay đổi (tức là khi ảnh được upload)
    image_input.change(
        fn=caption_image,
        inputs=[image_input],
        outputs=[caption_output, audio_output]
    )
demo.launch(share=True)

Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3204098626.py:81: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e39fa667fe5947b696.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
